In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Translate)=
# Translate

*Translating atomic coordinates along spatial displacement vectors.*

The function {func}`molsysmt.structure.translate` shifts atomic coordinates by applying 3D translation vectors across single structures or entire structural ensembles.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.structure.translate`.
:::


## Basic usage

Let's show how to translate Met-enkephalin along a 3D displacement vector $(\Delta x, \Delta y, \Delta z) = (1.0, 2.0, -0.5)\text{ nm}$:


In [2]:
import molsysmt as msm
import pyunitwizard as puw
import numpy as np


In [3]:
molsys = msm.convert(msm.systems['Met-enkephalin']['met_enkephalin.h5msm'])
molsys = msm.structure.center(molsys, selection='all')


:::{note} Demo Systems Catalog
:class: dropdown

This tutorial uses demonstration datasets provided by MolSysMT. To explore the full catalog of bundled systems, forms, and file paths, visit the {ref}`Demo Systems <user-foundations-entrance-demo-systems>` guide.
:::


We inspect the initial geometric center before displacement:


In [4]:
initial_center = msm.structure.get_center(molsys)
print('Initial center:', initial_center)


Initial center: [[[2.559680862330222e-16 1.603655480014115e-16 3.912765173592132e-17]]] nanometer


We apply the translation using {func}`molsysmt.structure.translate`:


In [5]:
shift_vector = puw.quantity([1.0, 2.0, -0.5], 'nm')
molsys_translated = msm.structure.translate(molsys, translation=shift_vector)


We verify that the new centroid matches the applied translation:


In [6]:
translated_center = msm.structure.get_center(molsys_translated)
print('Translated center:', translated_center)


Translated center: [[[1.0000000000000009 2.0 -0.5000000000000006]]] nanometer


## Translating a specific atom selection

Using the `selection` parameter, we can translate a specific molecular subset (such as a single residue or ligand) while leaving the rest of the system unperturbed:


In [7]:
molsys_sub = msm.structure.translate(molsys, selection='group_index==0', translation='[0.5, 0.0, 0.0] nm')
center_res0 = msm.structure.get_center(molsys_sub, selection='group_index==0')
center_res1 = msm.structure.get_center(molsys_sub, selection='group_index==1')
print('Center of translated residue 0:', center_res0)
print('Center of untranslated residue 1:', center_res1)


Center of translated residue 0: [[[-0.1740717991358705 -0.2432726191741132 0.0630981191633839]]] nanometer
Center of untranslated residue 1: [[[-0.40817815106776034 -0.30096779509432736 -0.023855374898798565]]] nanometer


## Structure-dependent translations across ensembles

When working with multi-structure systems, `translation` accepts arrays with shape `(n_structures, 1, 3)` to apply distinct displacements per structure (e.g. ensemble drift corrections or continuous displacements):


In [8]:
traj = msm.convert(msm.systems['pentalanine']['traj_pentalanine.h5msm'])
n_structures = msm.get(traj, n_structures=True)

frame_shifts = np.zeros((n_structures, 1, 3))
frame_shifts[:, 0, 0] = np.linspace(0.0, 2.0, n_structures)
traj_shifted = msm.structure.translate(traj, translation=puw.quantity(frame_shifts, 'nm'))

first_center = msm.structure.get_center(traj_shifted, structure_indices=0)
last_center = msm.structure.get_center(traj_shifted, structure_indices=n_structures - 1)
print('Center at structure 0:', first_center)
print('Center at final structure:', last_center)


Center at structure 0: [[[0.8281487274554468 1.1086905834174925 -0.007842551886795028]]] nanometer
Center at final structure: [[[2.8383589060075822 1.0765238024534718 0.0412316741797352]]] nanometer


:::{seealso} Related Tools & References
:class: dropdown

- {ref}`Convert <Tutorial_Convert>`: Convert molecular systems between different forms with {func}`molsysmt.basic.convert`.
- {ref}`Center <Tutorial_Center>`: Center molecular coordinates around reference points with {func}`molsysmt.structure.center`.
- {ref}`Get center <Tutorial_Get_center>`: Calculate geometric or mass-weighted centroids with {func}`molsysmt.structure.get_center`.
- {ref}`Rotate <Tutorial_Rotate>`: Apply 3D spatial rotations with {func}`molsysmt.structure.rotate`.
- {ref}`Move away <Tutorial_Move_away>`: Separate molecular entities along directional vectors with {func}`molsysmt.structure.move_away`.
:::
